# Part 2: Image Calibration Pipeline

This section calibrates our previously generated master images. You will be asked to fill in the given functions below. 

### Import Libraries

This assignment can be completed without the use of additional libraries.

In [ ]:
# given libraries and functions
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from astropy.io import fits
from astropy.stats import SigmaClip, sigma_clipped_stats
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from photutils.detection import DAOStarFinder
from photutils.background import Background2D, MedianBackground

## Vega Calibration

In this section we will use a Vega observation to calibrate the zero point for our cluster observation.

### Import Images

Hint: an easier way to open a FITS file along with its header is by using *fits.getdata(filename, header=True)*

In [ ]:
# open a single Vega frame for each filter


### Plot Vega

We should plot Vega through both filters to make sure we can see it properly.

Hint: use *axs[0]* to plot an image at the first subplot.

In [ ]:
# plot setup
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# plot R image

# plot G image


### Detect Vega

We can use DAOStarFinder to locate Vega in the image and retrieve its pixel coordinates.

Hint: get the median of the data using *sigma_clipped_stats(img)*

Hint: configure a DAOStarFinder object named daofind, *fwm=4* and *threshold=5\*std* is a good start

Hint: daofind() returns an Astropy Table, we should sort by the brightest source detected using *np.argmax(tablename["flux"])*

In [ ]:
#TODO: complete the following function

# locates the brightest source in an image
def detect_vega(img):
    
    # calculate image stats
    
    # use daofind to detect sources
    
    # retrieve brightest source
    
    # return (x, y) coordinates of detected source
    


# TODO: call the function for each image and store x and y coordinates


### Measure Vega

We now want to measure the flux of Vega in DN (counts).

We will use an aperture-annulus subtraction method, meaning that we measure within a radius on the star itself and use an annulus for the background estimation. Use the annulus to get an average background value per unit area, then multiply by the aperture area to obtain the total background level within the aperture.

Hint: use *CircularAperture* and *CircularAnnulus* centred on a position (x, y) and adjust the radius as needed.

Hint: use aperture_photometry() on the image and pass it an aperture or annulus to measure the counts within. You may want to pull the value from the table using the key *['aperture_sum']*

Tip: you may want to plot the aperture/annulus to confirm it fits correctly using .plot() on the object after displaying your image

In [ ]:
# TODO: complete the following function

# measures counts within an aperture
def measure_vega(img, x, y):
    
    # create Circles
    
    # measure within Circles

    # plot to confirm positioning
    
    # return background subtracted sum



# TODO: call the function for each image and store counts for each image


### Normalize Counts

We have our measured counts in DN, but we have yet to normalize to our exposure time. Pull the values from the image headers and calculate the counts per second for each filter.

Hint: use the key *['EXPTIME']* to access it within the header

In [ ]:
# fetch exposure time from image headers

# normalize counts


### Calculate Zero Point

You may recall that Vega has a magnitude of 0 in the Vega magnitude system. With this in mind and our known counts per second, we can no calculate a zero point correction for our observation. This can later be applied when scaling the magnitudes of the stars in our cluster in order to build an accurate CMD.

$$m = ZP - 2.5 \log_{10}\left(\frac{DN}{t}\right)$$



In [ ]:
# calculate zero point for each filter

# print values to reference later


## Background Calibration

Before we apply this correction to the stars in our image, we should first perform a background subtraction.

### Open Master Stacks

Open the master stacks generated in the previous assignment and store the images and headers.

In [ ]:
# store master image data and headers


### Mask Cluster Centre

The centre of our cluster is very bright and dense which may affect background and FWHM estimates. Let us create a mask in this area for each filter to use later on.

Hint: *img.shape* returns a tuple of (x, y) dimensions. Index 0 belongs to y (number of rows) and index 1 belongs to x (number of columns)

Hint: use *y, x = np.indices(dimensions)* to get arrays of all pixel coordinates

Hint: use the squared distance formula to calculate the distance from each pixel to the centre

$(x - x_c)^2 + (y - y_c)^2 \le r^2$

In [ ]:
# TODO: complete the following function

# generates a binary mask for an image within given radius
def create_mask(img, radius=300):

    # get image dimensions tuple
    
    # create arrays of all pixel coordinates
    
    # find image centre coordinates - replace 'None' with your own code
    x_c = None
    y_c = None
    
    # set up a True/False map of pixels within a given radius using the given formula
    
    # plot mask to confirm
    plt.figure(figsize=(8, 8))
    plt.imshow(img, origin="lower", cmap="gray", vmin=np.percentile(img, 5), vmax=np.percentile(img, 99.9))
    circle = Circle((x_c, y_c), radius, edgecolor="red", facecolor="red", alpha=0.2, linewidth=2)
    plt.gca().add_patch(circle)
    plt.title("M13 Background Mask")
    plt.show()
    
    # return pixel map mask
    


# TODO: call the function for each image and store the generated masks


### Background Subtraction

We can estimate the background of our image using a *Background2D* object. We can use the mask we just created to prevent the cluster core from affecting the median value. This will generate a clean image we can save for our star detection process in the next assignment.

Hint: store a model of the image background with *bg = Background2D()*, by passing it the image, *box_size=(50, 50)*, *sigma_clip=SigmaClip(sigma=3)*, *bkg_estimator=MedianBackground()*, and the relevant mask

Hint: use *bg.background* to get the background array model

In [ ]:
# TODO: complete following the function

# creates a clean background-subtracted image
def subtract_background(img, mask):
    
    # create Background2D object - replace 'None' with your own code
    bg = None
    
    # calculate the background values - replace 'None'
    bg_model = None
    
    # subtract background values from the original image - replace 'None'
    clean = None
    
    # this code will compare the original, background, and clean images
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(img, origin="lower", cmap="gray")
    axs[0].set_title("Original")
    axs[1].imshow(bg_model, origin="lower", cmap="gray")
    axs[1].set_title("Background Model")
    axs[2].imshow(clean, origin="lower", cmap="gray")
    axs[2].set_title("Background Subtracted")
    plt.show()
    
    # return the clean image



# TODO: call the function for each image and store the clean images
   

### Save Clean Images

We will need these clean images for the next assignment, so we should save them to a new FITS file.

In [ ]:
# create a new HDU 

# write to file
